In [1]:

import pandas as pd
import numpy as np
from sklearn.metrics import mean_squared_error
from math import sqrt
from statsmodels.tsa.arima.model import ARIMA
from prophet import Prophet


In [2]:

# Load dataset
file_path = "/content/AirQualityUCI.xlsx"
df = pd.read_excel(file_path, sheet_name='AirQualityUCI')

# Combine date and time
df['Datetime'] = pd.to_datetime(df['Date'].astype(str) + ' ' + df['Time'].astype(str), errors='coerce')
df = df.drop(columns=['Date', 'Time'])
df.set_index('Datetime', inplace=True)
df.replace(-200, np.nan, inplace=True)
df.sort_index(inplace=True)

# Drop NMHC(GT) due to excessive missingness and interpolate the rest
df = df.drop(columns=['NMHC(GT)'])
df = df.interpolate(method='linear').dropna()


In [3]:

# Define targets and thresholds
targets = {
    'CO(GT)': 10,
    'PT08.S1(CO)': 210,
    'C6H6(GT)': 6,
    'PT08.S2(NMHC)': 250,
    'NOx(GT)': 190,
    'PT08.S3(NOx)': 196.0619,
    'NO2(GT)': 120,
    'PT08.S4(NO2)': 300.7,
    'PT08.S5(O3)': 400.25,
    'T': 12,
    'RH': 18,
    'AH': 7
}

# Train-test split
n_validation = int(len(df) * 0.10)
train = df.iloc[:-n_validation]
test = df.iloc[-n_validation:]


In [4]:

# ARIMA Forecast
forecast_arima = pd.DataFrame(index=test.index)
rmse_arima_scores = {}

for col in targets.keys():
    y_train_col = train[col][-2000:]
    y_test_col = test[col]

    try:
        model = ARIMA(y_train_col, order=(5, 1, 0))
        results = model.fit()
        forecast = results.forecast(steps=n_validation)
        forecast_arima[col] = forecast
        rmse = sqrt(mean_squared_error(y_test_col, forecast))
        rmse_arima_scores[col] = rmse
    except Exception as e:
        rmse_arima_scores[col] = f"ARIMA Error: {str(e)}"
        forecast_arima[col] = np.nan


/usr/local/lib/python3.11/dist-packages/statsmodels/tsa/base/tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency h will be used.
  self._init_dates(dates, freq)
/usr/local/lib/python3.11/dist-packages/statsmodels/tsa/base/tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency h will be used.
  self._init_dates(dates, freq)
/usr/local/lib/python3.11/dist-packages/statsmodels/tsa/base/tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency h will be used.
  self._init_dates(dates, freq)
/usr/local/lib/python3.11/dist-packages/statsmodels/tsa/base/tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency h will be used.
  self._init_dates(dates, freq)
/usr/local/lib/python3.11/dist-packages/statsmodels/tsa/base/tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency h will be used.
  self._init_dates(dat

In [5]:

# Prophet Forecast
forecast_prophet = pd.DataFrame(index=test.index)
rmse_prophet_scores = {}

for col in targets.keys():
    y_train_df = train[col][-2000:].reset_index().rename(columns={"Datetime": "ds", col: "y"})
    y_test_col = test[col]

    try:
        model = Prophet(daily_seasonality=True, weekly_seasonality=True)
        model.fit(y_train_df)

        future = model.make_future_dataframe(periods=n_validation, freq='H')
        forecast = model.predict(future)
        forecast_tail = forecast[['ds', 'yhat']].set_index('ds').iloc[-n_validation:]
        forecast_prophet[col] = forecast_tail['yhat'].values
        rmse = sqrt(mean_squared_error(y_test_col, forecast_tail['yhat']))
        rmse_prophet_scores[col] = rmse
    except Exception as e:
        rmse_prophet_scores[col] = f"Prophet Error: {str(e)}"
        forecast_prophet[col] = np.nan


INFO:prophet:Disabling yearly seasonality. Run prophet with yearly_seasonality=True to override this.
DEBUG:cmdstanpy:input tempfile: /tmp/tmpx5mctxml/1463besr.json
DEBUG:cmdstanpy:input tempfile: /tmp/tmpx5mctxml/csw3ym6p.json
DEBUG:cmdstanpy:idx 0
DEBUG:cmdstanpy:running CmdStan, num_threads: None
DEBUG:cmdstanpy:CmdStan args: ['/usr/local/lib/python3.11/dist-packages/prophet/stan_model/prophet_model.bin', 'random', 'seed=51070', 'data', 'file=/tmp/tmpx5mctxml/1463besr.json', 'init=/tmp/tmpx5mctxml/csw3ym6p.json', 'output', 'file=/tmp/tmpx5mctxml/prophet_modelpf2tpr5g/prophet_model-20250430192220.csv', 'method=optimize', 'algorithm=lbfgs', 'iter=10000']
19:22:20 - cmdstanpy - INFO - Chain [1] start processing
INFO:cmdstanpy:Chain [1] start processing
19:22:20 - cmdstanpy - INFO - Chain [1] done processing
INFO:cmdstanpy:Chain [1] done processing
/usr/local/lib/python3.11/dist-packages/prophet/forecaster.py:1854: FutureWarning: 'H' is deprecated and will be removed in a future version

In [7]:

# Save forecasts and RMSE to Excel
with pd.ExcelWriter("forecast_submission.xlsx") as writer:
    forecast_arima.to_excel(writer, sheet_name="ARIMA_Forecasts")
    forecast_prophet.to_excel(writer, sheet_name="Prophet_Forecasts")
    pd.DataFrame(rmse_arima_scores.items(), columns=["Variable", "RMSE"]).to_excel(writer, sheet_name="ARIMA_RMSE", index=False)
    pd.DataFrame(rmse_prophet_scores.items(), columns=["Variable", "RMSE"]).to_excel(writer, sheet_name="Prophet_RMSE", index=False)
